# **1. Data Download and Paths Configuration**

In [ ]:
from google.colab import drive
import subprocess
import zipfile
import os

In [ ]:
# Mount Google Drive so that datasets, checkpoints, and experiment outputs
# persist across Colab sessions. force_remount ensures a clean mount even
# if the session was interrupted mid-run.

drive.mount("/content/drive", force_remount = True)

In [ ]:
# Base path of the user's Google Drive inside the Colab container.
# All project paths below are derived from this variable so that a single
# change here propagates everywhere.

GDRIVE = "/content/drive/MyDrive"

In [ ]:
# Root directory of the AVION pre-processed EK-100 videos.
# Videos are stored as 320p / 15-second chunks at 30 fps encoded with libx264.
# The folder must contain one sub-directory per participant: P01/, P02/, ..., P37/

DATA_ROOT = f"{GDRIVE}/EK100_320p_15sec_30fps_libx264"

In [ ]:
# Root directory of the modified SMS Loss repository. 
# This version of the repository have all the needed patches
# to fix bugs and ensure the correct execution of the training and inference. 

SMS_ROOT = f"{GDRIVE}/SMS_Loss_Custom"

In [ ]:
# Directory for EK-100 retrieval annotation CSVs and relevancy pickle files.
# Upload these from the local EK100_MIR/data/ folder before running,
# or let the download cells below fetch them automatically.

ANNOT_DIR = f"{GDRIVE}/EK100_annotations"

In [ ]:
# Output directory for checkpoints, logs, and evaluation results.
# Saved to Drive so that runs survive Colab timeouts.

EXP_DIR = f"{GDRIVE}/experiments/sms_vitl"

In [ ]:
# Path where the AVION ViT-L pretrained checkpoint will be stored.
# If the file is not present a later cell downloads it automatically from
# the UT Austin Box link provided by the AVION authors.

PRETRAIN_CKPT = f"{GDRIVE}/checkpoints/avion_pretrain_lavila_vitl_best.pt"

In [ ]:
# Create the output directories on Drive if they don't already exist,
# then print each path and whether the data root was found.

os.makedirs(ANNOT_DIR, exist_ok = True)
os.makedirs(EXP_DIR, exist_ok = True)
os.makedirs(os.path.dirname(PRETRAIN_CKPT), exist_ok = True)

print(f"DATA_ROOT : {DATA_ROOT}")
print(f"  exists  : {os.path.isdir(DATA_ROOT)}")
print(f"ANNOT_DIR : {ANNOT_DIR}")
print(f"EXP_DIR   : {EXP_DIR}")

In [ ]:
# Validate that the EK-100 video data is accessible.
# The AVION GDrive share can arrive either as a mounted folder shortcut
# or as a single zip file — this cell handles both cases.

if os.path.isdir(DATA_ROOT):
    # Happy path: folder already exists, list the first few participant dirs.
    participants = [d for d in os.listdir(DATA_ROOT) if d.startswith('P')]
    print(f"Found {len(participants)} participant folders: {sorted(participants)[:5]}...")

else:
    print("DATA_ROOT not found as a directory.")
    # Fallback: check whether the GDrive shortcut landed as a zip archive.
    zip_path = f"{GDRIVE}/EK100_320p_15sec_30fps_libx264.zip"

    if os.path.isfile(zip_path):
        print(f"Found zip at {zip_path}. Extracting to {GDRIVE}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(GDRIVE)
        print("Extraction complete!")

        # Verify the extraction produced the expected folder structure.
        if os.path.isdir(DATA_ROOT):
            participants = [d for d in os.listdir(DATA_ROOT) if d.startswith('P')]
            print(f"Found {len(participants)} participant folders: {sorted(participants)[:5]}...")
        else:
            print("Extraction done but DATA_ROOT still not found. Check that the zip extracts to the expected folder name.")
    else:
        # Neither folder nor zip present — the user must add the GDrive shortcut first.
        print("Neither folder nor zip found. Please add the GDrive shortcut first.")
        print("Link: https://drive.google.com/file/d/13J2uC2g2H_DEHrBvgr5Aiu0BgqlCvWqG/view")

In [ ]:
# Validate that the SMS Loss Custom Repository is accessible. 
# This cell handles the case when the repository is alredy extracted. 
# Also handles the case where there is the zip file and need to be extracted. 

if os.path.isdir(SMS_ROOT):
    # Happy path: folder already exists, list the first few participant dirs.
    contents = os.listdir(SMS_ROOT)
    print(f"SMS_Loss_Custom repository contents ({len(contents)} items):")

else:
    print("SMS_ROOT not found as a directory.")
    # Fallback: check whether the GDrive shortcut landed as a zip archive.
    zip_path = f"{GDRIVE}/SMS_Loss_Custom"

    if os.path.isfile(zip_path):
        print(f"Found zip at {zip_path}. Extracting to {GDRIVE}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(GDRIVE)
        print("Extraction complete!")

        # Verify the extraction produced the expected folder structure.
        if os.path.isdir(SMS_ROOT):
            contents = os.listdir(SMS_ROOT)
            print(f"SMS_Loss_Custom repository contents ({len(contents)} items):")
        else:
            print("Extraction done but SMS_ROOT still not found. Check that the zip extracts to the expected folder name.")
    else:
        # Neither folder nor zip present — the user must add the GDrive shortcut first.
        print("Neither folder nor zip found. Please add the zip file to GDrive first.")

In [ ]:
# Download the AVION LaViLa ViT-L pretrain checkpoint if it is not already on Drive.
# Source: https://github.com/zhaoyue-zephyrus/AVION/blob/main/scripts/download_checkpoints.sh
# Download takes a few minutes depending on Colab bandwidth.

if not os.path.isfile(PRETRAIN_CKPT):
    print("Downloading AVION ViT-L pretrain checkpoint...")
    !wget --show-progress -O "$PRETRAIN_CKPT" \
        "https://utexas.box.com/shared/static/1iatmrs7ufdeooce09a61t1n6wsouf4l.pt"
else:
    print(f"Checkpoint already present ({os.path.getsize(PRETRAIN_CKPT)/1e9:.2f} GB).")

In [ ]:
# Define paths for the four annotation files needed for training and evaluation:
#   TRAIN_CSV  — video-caption pairs for the retrieval training split
#   TEST_CSV   — video-caption pairs for the retrieval test split
#   TRAIN_REL  — pairwise relevancy scores for the training split (used for SMS loss)
#   TEST_REL   — pairwise relevancy scores for the test split (used for mAP / nDCG)
#
# If any file is missing, the cells below download it automatically from
# the EPIC-KITCHENS GitHub annotations repo.

TRAIN_CSV = f"{ANNOT_DIR}/EPIC_100_retrieval_train.csv"
TEST_CSV  = f"{ANNOT_DIR}/EPIC_100_retrieval_test.csv"
TRAIN_REL = f"{ANNOT_DIR}/caption_relevancy_EPIC_100_retrieval_train.pkl"

In [ ]:
# Resolve the path for the test-split relevancy pickle.
# The AVION GDrive archive bundles it inside the epic-kitchens annotations sub-folder;
# if that path exists we use it directly, otherwise we fall back to ANNOT_DIR where
# the user may have manually uploaded it.

TEST_REL_AVION = (
    f"{DATA_ROOT}/epic-kitchens-100-annotations/"
    "retrieval_annotations/relevancy/"
    "caption_relevancy_EPIC_100_retrieval_test.pkl"
)
TEST_REL = TEST_REL_AVION if os.path.isfile(TEST_REL_AVION) else f"{ANNOT_DIR}/caption_relevancy_EPIC_100_retrieval_test.pkl"

# Base URL for all EPIC-KITCHENS 100 annotation files on GitHub.
BASE = "https://raw.githubusercontent.com/epic-kitchens/epic-kitchens-100-annotations/master/retrieval_annotations"

In [ ]:
# Download the train and test CSV files if they are not already present.
# Each CSV contains (video_id, narration_id, start_frame, stop_frame, narration) columns
# used to build the retrieval dataset.

for path, url in [
    (TRAIN_CSV, f"{BASE}/EPIC_100_retrieval_train.csv"),
    (TEST_CSV,  f"{BASE}/EPIC_100_retrieval_test.csv"),
]:
    if not os.path.isfile(path):
        print(f"Downloading {os.path.basename(path)} ...")
        subprocess.run(["wget", "-q", "-O", path, url], check=True)

In [ ]:
# Download the training-split relevancy pickle if missing.
# This file encodes soft relevancy labels between every query-gallery pair
# and is required by the SMS loss during fine-tuning.

BASE = "https://dl.fbaipublicfiles.com/lavila/metadata/EK100"

if not os.path.isfile(TRAIN_REL):
    url = f"{BASE}/caption_relevancy_EPIC_100_retrieval_train.pkl"
    print("Downloading train relevancy ...")
    subprocess.run(["wget", "-q", "-O", TRAIN_REL, url], check=True)
    print("Done!")
else:
    print(f"Already exists: {TRAIN_REL}")

In [ ]:
# Download the test-split relevancy pickle if missing.
# This file isnt hosted and need to be upload manually in the correct folder. 

if not os.path.isfile(TEST_REL):
    print(f"{TEST_REL} not foun")
    print("Upload it manually!")
else:
    print(f"Already exists: {TEST_REL}")

In [ ]:
# Sanity check: print True/False for each required annotation file.
# All four must be True before launching training.

print(f"TRAIN_CSV  : {os.path.isfile(TRAIN_CSV)}")
print(f"TEST_CSV   : {os.path.isfile(TEST_CSV)}")
print(f"TRAIN_REL  : {os.path.isfile(TRAIN_REL)}")
print(f"TEST_REL   : {os.path.isfile(TEST_REL)}  (path: {TEST_REL})")

In [ ]:
ANNOT_DIR = "/content/drive/MyDrive/EK100_annotations"
BASE = "https://raw.githubusercontent.com/epic-kitchens/epic-kitchens-100-annotations/master/retrieval_annotations"

# Download the sentence-level annotation CSVs if they are not present.
# These files map each video segment to a free-form sentence description
# and are used as the text queries during retrieval evaluation.

for fname in ["EPIC_100_retrieval_train_sentence.csv", "EPIC_100_retrieval_test_sentence.csv"]:
    path = f"{ANNOT_DIR}/{fname}"

    if not os.path.isfile(path):
        subprocess.run(["wget", "-q", "-O", path, f"{BASE}/{fname}"], check=True)
        print(f"Downloaded {fname}")
    else:
        print(f"Already present: {fname}")